# B2-023 — Practice p11

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** core · **Time budget:** 50 minutes  
**Concepts:** denoising-diffusion-probabilistic-models

## Task

Implement one reverse step with $\sigma_t^2=\tilde\beta_t$ and the full ancestral sampling loop with fixed noise.

**Contract.** Include your p09 `make_schedule` (same contract). `p_sample(model, x_t, t, z, schedule)` takes a Python int `t` in $1,\dots,T$ shared by the batch, calls `eps_hat = model(x_t, torch.full((B, 1), t / T, dtype=x_t.dtype))`, forms `mean = (x_t - beta[t] / sqrt(1 - alpha_bar[t]) * eps_hat) / sqrt(alpha[t])` with the coefficients taken as Python floats (`.item()`) from the float64 schedule, and returns `mean + sqrt(beta_tilde[t]) * z` for $t\ge2$ and exactly `mean` for $t=1$. `sample_loop(model, x_T, noises, schedule)` starts from `x = x_T` and, for `t = T, T-1, ..., 1` in that descending order, sets `x = p_sample(model, x, t, noises[t - 1], schedule)`; `noises` has shape `(T,B,d)`, `noises[0]` is ignored, and neither function draws random numbers or modifies its inputs in place.
Both functions return tensors with `x_t`'s (respectively `x_T`'s) shape and dtype; `p_sample` raises `ValueError` unless $1\le t\le T$, and `sample_loop` raises `ValueError` unless `noises.shape == (T, *x_T.shape)`.

**Banned:** drawing random numbers inside either function; `torch.distributions`; in-place modification of `x_T` or `noises`.

**Probes** (`schedule = make_schedule(5, 0.1, 0.5)`, the supplied `ToyEps` model, the literals below, `ATOL = RTOL = 1e-5`):

1. `p_sample(ToyEps(), XT2, 5, Z2, schedule)` equals `[[0.2857176960, -2.2329156399], [0.7895655632, 0.5145742893]]` and `p_sample(ToyEps(), XT2, 2, Z2, schedule)` equals `[[0.4609285593, -1.1616884470], [1.0526943207, 0.3654916286]]`.
2. **Final step.** `p_sample(ToyEps(), XT2, 1, Z2, schedule)` equals `[[0.2883037031, -0.7766073942], [0.9982443452, -0.0666666701]]` and is `torch.equal` to the result with `torch.zeros_like(Z2)` in place of `Z2`. A local variant that adds `sqrt(beta[1]) * z` at $t=1$ differs from it by more than `0.1`.
3. `sample_loop(ToyEps(), X_T, NOISES, schedule)` equals `[[-0.4030713737, -2.0024237633], [-1.2903687954, 0.1871905029]]`. A recording model shows the timesteps were visited as `t_scaled` values `1.0, 0.8, 0.6, 0.4, 0.2`, in that order. A local variant that visits the timesteps in ascending order differs by more than `0.1`.
4. Replacing `NOISES[0]` by `100 * torch.ones(2, 2)` leaves the output of `sample_loop` `torch.equal` to probe 3's, while replacing `NOISES[4]` the same way changes it.
5. `X_T` and `NOISES` are `torch.equal` to their clones after the call, and the invalid `t = 0`, `t = 6`, and a `noises` tensor of shape `(4,2,2)` each raise `ValueError`.

Explain in two sentences why the final step must be noiseless under this unit's convention.

**Required answer-check assertions.** Probes 1–5.

In [ ]:
import math
import torch
from torch import nn

XT2 = torch.tensor([[0.4, -0.8], [1.2, 0.0]])
Z2 = torch.tensor([[1.0, -1.0], [0.5, 2.0]])
X_T = torch.tensor([[1.0, -1.0], [0.0, 2.0]])
NOISES = torch.tensor([[[0.1, 0.2], [0.3, -0.1]],
                       [[-0.5, 0.4], [0.2, 0.2]],
                       [[1.0, -1.0], [0.0, 0.5]],
                       [[0.3, 0.3], [-0.3, 0.1]],
                       [[-0.2, 0.6], [0.4, -0.4]]])


class ToyEps(nn.Module):
    def forward(self, x_t, t_scaled):
        return 0.5 * x_t + t_scaled


ATOL = 1e-5
RTOL = 1e-5

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.